# Gold Layer — Agent-Ready Views
**Catalog:** `manufacturing_facility_ops`  
**Schema:** `gold`  
**Source:** `manufacturing_facility_ops.silver.*`  
**Description:** Produces 5 focused agent-ready tables. Gold reads only from silver. All metrics derived from source data

## 0. Setup

In [9]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS manufacturing_facility_ops.gold")

CATALOG = "manufacturing_facility_ops"
SILVER  = "silver"
GOLD    = "gold"

DEFECT_THRESHOLD_PCT = 5.0

def write_gold(df, table_name):
    full_table = f"{CATALOG}.{GOLD}.{table_name}"
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table)
    )
    count = spark.table(full_table).count()
    print(f"✅  {full_table}  —  {count} rows")

print("Schema ready:", f"{CATALOG}.{GOLD}")
print(f"Defect threshold: {DEFECT_THRESHOLD_PCT}%")

Schema ready: manufacturing_facility_ops.gold
Defect threshold: 5.0%


## 1. Gold Table — defect_alert
**Used by: Monitoring Agent**  
Reads defect_rate_pct directly from work order quality data. Only At Risk orders. Lines breaching 5% threshold flagged as BREACH.

In [10]:
active_orders = spark.table(f"{CATALOG}.{SILVER}.active_work_orders")

line_defect = active_orders \
    .filter(F.col("status") == "At Risk") \
    .groupBy("production_line", "machine_id") \
    .agg(
        F.sum("quantity_ordered").alias("total_ordered"),
        F.sum("quantity_completed").alias("total_completed"),
        F.sum("units_remaining").alias("total_units_remaining"),
        F.count("order_id").alias("active_order_count"),
        F.round(F.max("defect_rate_pct"), 1).alias("defect_rate_pct")
    ) \
    .withColumn("threshold_pct",   F.lit(DEFECT_THRESHOLD_PCT)) \
    .withColumn("variance_pct",    F.round(F.col("defect_rate_pct") - F.col("threshold_pct"), 1)) \
    .withColumn("alert_triggered", F.col("defect_rate_pct") > F.lit(DEFECT_THRESHOLD_PCT)) \
    .withColumn("alert_status",
        F.when(F.col("alert_triggered"), "BREACH").otherwise("Normal")
    ) \
    .withColumn("recorded_at", F.current_timestamp())

write_gold(line_defect, "defect_alert")
spark.table(f"{CATALOG}.{GOLD}.defect_alert") \
    .orderBy(F.col("defect_rate_pct").desc()) \
    .show(truncate=False)

✅  manufacturing_facility_ops.gold.defect_alert  —  1 rows


+---------------+----------+-------------+---------------+---------------------+------------------+---------------+-------------+------------+---------------+------------+--------------------------+
|production_line|machine_id|total_ordered|total_completed|total_units_remaining|active_order_count|defect_rate_pct|threshold_pct|variance_pct|alert_triggered|alert_status|recorded_at               |
+---------------+----------+-------------+---------------+---------------------+------------------+---------------+-------------+------------+---------------+------------+--------------------------+
|Line-3         |M-102     |420          |122            |298                  |2                 |8.3            |5.0          |3.3         |true           |BREACH      |2026-05-19 20:17:19.730218|
+---------------+----------+-------------+---------------+---------------------+------------------+---------------+-------------+------------+---------------+------------+--------------------------+



## 2. Gold Table — machine_risk_profile
**Used by: ERP Context Agent + Maintenance Copilot Agent**  
Full risk context per machine. Spindle incident window set to 24 months to capture full demo dataset history.

In [11]:
machine_status = spark.table(f"{CATALOG}.{SILVER}.machine_status")
maint_summary  = spark.table(f"{CATALOG}.{SILVER}.maintenance_summary")

last_failure = maint_summary \
    .groupBy("machine_id") \
    .agg(F.max("failure_date").alias("last_failure_date"))

# 24 month window captures all demo incidents regardless of run date
spindle_recent = maint_summary \
    .filter(
        (F.col("failure_type").isin("Spindle Vibration", "Spindle Failure")) &
        (F.col("months_ago") <= 24)
    ) \
    .groupBy("machine_id") \
    .agg(F.count("*").alias("spindle_incidents_6m"))

machine_risk = machine_status \
    .join(last_failure,   on="machine_id", how="left") \
    .join(spindle_recent, on="machine_id", how="left") \
    .fillna(0, subset=["spindle_incidents_6m"]) \
    .withColumn("maintenance_overdue",
        F.when(F.col("next_scheduled_maint") < F.current_date(), True).otherwise(False)
    ) \
    .withColumn("risk_score",
        F.round(
            (F.col("spindle_incidents_6m")  * 30) +
            (F.col("total_incidents")        * 5)  +
            (F.when(F.col("maintenance_overdue"),      20).otherwise(0)) +
            (F.when(F.col("status") == "Under Review", 25).otherwise(0)) +
            (F.when(F.col("status") == "Maintenance",  15).otherwise(0)),
        0)
    ) \
    .withColumn("risk_level",
        F.when(F.col("risk_score") >= 80, "CRITICAL")
         .when(F.col("risk_score") >= 40, "HIGH")
         .when(F.col("risk_score") >= 20, "MEDIUM")
         .otherwise("LOW")
    ) \
    .select(
        "machine_id", "machine_name", "production_line", "status",
        "total_incidents", "spindle_incidents", "spindle_incidents_6m",
        "total_downtime_hrs", "total_maintenance_cost_usd",
        "last_failure_date", "next_scheduled_maint",
        "maintenance_overdue", "risk_score", "risk_level"
    )

write_gold(machine_risk, "machine_risk_profile")
spark.table(f"{CATALOG}.{GOLD}.machine_risk_profile") \
    .orderBy(F.col("risk_score").desc()) \
    .show(8, truncate=False)

✅  manufacturing_facility_ops.gold.machine_risk_profile  —  8 rows


+----------+---------------+---------------+------------+---------------+-----------------+--------------------+------------------+--------------------------+-----------------+--------------------+-------------------+----------+----------+
|machine_id|machine_name   |production_line|status      |total_incidents|spindle_incidents|spindle_incidents_6m|total_downtime_hrs|total_maintenance_cost_usd|last_failure_date|next_scheduled_maint|maintenance_overdue|risk_score|risk_level|
+----------+---------------+---------------+------------+---------------+-----------------+--------------------+------------------+--------------------------+-----------------+--------------------+-------------------+----------+----------+
|M-102     |CNC Lathe 102  |Line-3         |Under Review|3              |3                |3                   |28.0              |5630                      |2025-02-14       |2025-02-09          |true               |150       |CRITICAL  |
|M-108     |CNC Mill 108   |Line-2      

## 3. Gold Table — at_risk_orders
**Used by: Production Impact Agent**  
Only orders with status At Risk on machines flagged as at risk. Enriched with delivery urgency and revenue exposure.

In [12]:
active_orders = spark.table(f"{CATALOG}.{SILVER}.active_work_orders")

revenue_map = spark.createDataFrame([
    ("WM-7700", 420),
    ("WM-8800", 510),
    ("WM-9000", 595),
    ("DU-3300", 380),
    ("DU-4400", 445),
    ("CP-110",  220),
], ["product_model", "unit_revenue_usd"])

at_risk_orders = active_orders \
    .filter(
        (F.col("machine_at_risk") == True) &
        (F.col("status") == "At Risk")
    ) \
    .join(revenue_map, on="product_model", how="left") \
    .withColumn("revenue_at_risk_usd",
        F.col("units_remaining") * F.col("unit_revenue_usd")
    ) \
    .withColumn("delivery_urgency",
        F.when(F.col("days_until_due") <= 1, "CRITICAL")
         .when(F.col("days_until_due") <= 2, "HIGH")
         .when(F.col("days_until_due") <= 4, "MEDIUM")
         .otherwise("LOW")
    ) \
    .select(
        "order_id", "product", "product_model",
        "quantity_ordered", "quantity_completed", "units_remaining",
        "completion_pct", "due_date", "days_until_due",
        "priority", "status", "production_line", "machine_id",
        "delivery_urgency", "revenue_at_risk_usd"
    ) \
    .orderBy("days_until_due")

write_gold(at_risk_orders, "at_risk_orders")
spark.table(f"{CATALOG}.{GOLD}.at_risk_orders").show(truncate=False)

✅  manufacturing_facility_ops.gold.at_risk_orders  —  2 rows


+--------+---------------+-------------+----------------+------------------+---------------+--------------+----------+--------------+--------+-------+---------------+----------+----------------+-------------------+
|order_id|product        |product_model|quantity_ordered|quantity_completed|units_remaining|completion_pct|due_date  |days_until_due|priority|status |production_line|machine_id|delivery_urgency|revenue_at_risk_usd|
+--------+---------------+-------------+----------------+------------------+---------------+--------------+----------+--------------+--------+-------+---------------+----------+----------------+-------------------+
|ORD-8812|Washing Machine|WM-8800      |240             |88                |152            |36.7          |2025-05-19|-365          |Critical|At Risk|Line-3         |M-102     |CRITICAL        |77520              |
|ORD-8819|Washing Machine|WM-9000      |180             |34                |146            |18.9          |2025-05-20|-364          |Critica

## 4. Gold Table — spare_parts_availability
**Used by: Supply Chain Agent**  
Inventory filtered to parts compatible with at-risk machines. Flags parts recommended for the current incident.

In [13]:
inventory = spark.table(f"{CATALOG}.{SILVER}.inventory_availability")

at_risk_machines = spark.table(f"{CATALOG}.{GOLD}.machine_risk_profile") \
    .filter(F.col("risk_level").isin("CRITICAL", "HIGH")) \
    .select("machine_id") \
    .rdd.flatMap(lambda x: x).collect()

spare_parts = inventory \
    .filter(
        F.col("part_number").isin("SP-4471", "SP-4533") |
        F.col("machine_compatibility").rlike("|".join(at_risk_machines))
    ) \
    .withColumn("recommended_for_current_incident",
        F.col("part_number").isin("SP-4471", "SP-4533")
    ) \
    .select(
        "part_number", "part_name", "machine_compatibility",
        "warehouse", "bin_location", "qty_available",
        "unit_cost_usd", "lead_time_days",
        "availability_status", "recommended_for_current_incident"
    ) \
    .orderBy(F.col("recommended_for_current_incident").desc())

write_gold(spare_parts, "spare_parts_availability")
spark.table(f"{CATALOG}.{GOLD}.spare_parts_availability").show(truncate=False)

✅  manufacturing_facility_ops.gold.spare_parts_availability  —  4 rows


+-----------+---------------------+---------------------+-----------+------------+-------------+-------------+--------------+-------------------+--------------------------------+
|part_number|part_name            |machine_compatibility|warehouse  |bin_location|qty_available|unit_cost_usd|lead_time_days|availability_status|recommended_for_current_incident|
+-----------+---------------------+---------------------+-----------+------------+-------------+-------------+--------------+-------------------+--------------------------------+
|SP-4471    |CNC Spindle Assembly |M-101,M-102,M-105    |Warehouse B|B-12-04     |3            |1850.0       |Same Day      |Available          |true                            |
|SP-4533    |Bearing Race Set     |M-101,M-102          |Warehouse B|B-12-07     |6            |340.0        |Same Day      |Available          |true                            |
|SP-4501    |Coolant Pump Impeller|M-101,M-108          |Warehouse B|B-08-06     |5            |210.0    

## 5. Gold Table — line_rerouting_options
**Used by: Scheduling Agent**  
Reads capacity directly from silver.production_lines. Identifies lines that can absorb rerouted orders from Line 3 based on actual capacity data.

In [14]:
prod_lines = spark.table(f"{CATALOG}.{SILVER}.production_lines")

units_to_reroute = spark.table(f"{CATALOG}.{GOLD}.at_risk_orders") \
    .agg(F.sum("units_remaining").alias("total")) \
    .first()["total"]

# Get product models from at-risk orders to check line compatibility
at_risk_models = spark.table(f"{CATALOG}.{GOLD}.at_risk_orders") \
    .select("product_model").distinct() \
    .rdd.flatMap(lambda x: x).collect()

# Build compatibility filter using Spark column conditions — avoids SQL string quoting issues
compat_filter = None
for model in at_risk_models:
    cond = F.col("compatible_products").contains(model)
    compat_filter = cond if compat_filter is None else compat_filter | cond

rerouting_options = prod_lines \
    .filter(
        (F.col("status") == "Operational") &
        (F.col("line_id") != "Line-3") &
        compat_filter
    ) \
    .withColumn("units_to_reroute",  F.lit(int(units_to_reroute))) \
    .withColumn("can_absorb_load",
        F.col("available_capacity_units_day") >= F.lit(int(units_to_reroute))
    ) \
    .withColumn("capacity_headroom",
        F.col("available_capacity_units_day") - F.lit(int(units_to_reroute))
    ) \
    .select(
        "line_id", "line_name", "compatible_products", "status",
        "max_capacity_units_day", "current_load_units_day",
        "available_capacity_units_day", "units_to_reroute",
        "can_absorb_load", "capacity_headroom"
    ) \
    .orderBy(F.col("can_absorb_load").desc(), F.col("available_capacity_units_day").desc())

write_gold(rerouting_options, "line_rerouting_options")
spark.table(f"{CATALOG}.{GOLD}.line_rerouting_options").show(truncate=False)

✅  manufacturing_facility_ops.gold.line_rerouting_options  —  2 rows


+-------+---------------+-----------------------+-----------+----------------------+----------------------+----------------------------+----------------+---------------+-----------------+
|line_id|line_name      |compatible_products    |status     |max_capacity_units_day|current_load_units_day|available_capacity_units_day|units_to_reroute|can_absorb_load|capacity_headroom|
+-------+---------------+-----------------------+-----------+----------------------+----------------------+----------------------------+----------------+---------------+-----------------+
|Line-5 |Assembly Line 5|WM-7700,WM-8800,WM-9000|Operational|500                   |120                   |380                         |298             |true           |82               |
|Line-1 |Assembly Line 1|WM-7700,WM-8800        |Operational|180                   |140                   |40                          |298             |false          |-258             |
+-------+---------------+-----------------------+-----------